# TFM — Sistema de Apoyo a la Clasificación de FSE+
## 02: Depuración y Extracción del Corpus Maestro (Data Preparation)
---
### Objetivo del Módulo
El objetivo es transformar cientos de miles de transacciones ruidosas en un único ecosistema de inferencia.
Acatando la decisión arquitectónica (NLP Puro), la Inteligencia Artificial aprenderá **excluyentemente** el patrón estadístico entre texto (`DESCRIPCION_ARTICULO`) y etiqueta (`ESTADO`).
El *Feature Engineering* clásico numérico (precios) es descartado para este clasificador cognitivo.

**Cada decisión de limpieza está respaldada por un hallazgo medible del EDA Técnico (`01b`).**

In [1]:
import pandas as pd
import numpy as np
import re
import json

# === CARGA DEL DATASET ===
# El CSV usa separador ';' y comas decimales (formato europeo)
# Hallazgo 01b: La estructura del fichero requiere sep=';' obligatorio
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/TFM - Master en IA y Ciencia de Datos/data/raw'
OUTPUT_PATH = '/content/drive/MyDrive/Colab Notebooks/TFM - Master en IA y Ciencia de Datos/data/processed'

import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

try:
    df_raw = pd.read_csv(f'{DATA_PATH}/Dataset.csv', sep=';', dtype={'COD_EAN': str}, low_memory=False)
    print(f'\U0001F539 [OK] Dataset Cargado: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas.')
except FileNotFoundError:
    raise FileNotFoundError('\u26A0\uFE0F Sube el fichero Dataset.csv al entorno de Colab.')

Mounted at /content/drive
🔹 [OK] Dataset Cargado: 659,272 filas x 17 columnas.


## 1. Normalización Léxica Profunda

**Justificación (EDA 01b — Sección 4, Indicadores de Ruido):**
El análisis técnico reveló que un porcentaje significativo de las descripciones contienen:
- Puntos de abreviatura (`DET. LIQ.`) que fragmentan tokens idénticos en el vectorizador TF-IDF.
- Barras (`LECHE E/D`) que generan tokens espurios.
- Espacios múltiples que duplican artificialmente las entradas únicas.

La normalización aplica las siguientes transformaciones conservadoras:
1. Conversión a mayúsculas y eliminación de espacios laterales.
2. Eliminación de puntos y barras redundantes.
3. Colapso de espacios múltiples a uno solo.

> **Nota:** No se eliminan dígitos ni stop words en esta fase. Esa decisión se delega al vectorizador (TF-IDF con `min_df`) para no perder información potencialmente discriminante.

In [2]:
df = df_raw.copy()

# Columnas de trabajo
columnas_base = ['NOMBRE_EMPRESA', 'DESCRIPCION_ARTICULO', 'ESTADO', 'SUBFAMILIA']

# Verificar que las columnas existen
for col in columnas_base:
    assert col in df.columns, f'Columna faltante: {col}'

# Eliminar filas sin texto o sin etiqueta (no hay nada que aprender de ellas)
n_antes = len(df)
df = df.dropna(subset=['DESCRIPCION_ARTICULO', 'ESTADO'])
n_nulos_eliminados = n_antes - len(df)
print(f'Filas eliminadas por nulos en DESCRIPCION_ARTICULO o ESTADO: {n_nulos_eliminados:,}')

# Seleccionar columnas de trabajo
df = df[columnas_base].copy()

# --- Normalización del texto ---
df['ESTADO'] = df['ESTADO'].astype(str).str.upper().str.strip()

def normalizar_descripcion(texto):
    """Normalización conservadora del texto de ticket."""
    t = str(texto).upper().strip()
    t = t.replace('.', ' ')   # Puntos de abreviatura -> espacio
    t = t.replace('/', ' ')   # Barras logísticas -> espacio
    t = re.sub(r'\s+', ' ', t)  # Colapsar espacios múltiples
    return t.strip()

df['DESCRIPCION_ARTICULO'] = df['DESCRIPCION_ARTICULO'].apply(normalizar_descripcion)

print(f'\U0001F538 Normalización Profunda Completada.')
print(f'   Ejemplo: "{df_raw["DESCRIPCION_ARTICULO"].iloc[0]}" -> "{df["DESCRIPCION_ARTICULO"].iloc[0]}"')

Filas eliminadas por nulos en DESCRIPCION_ARTICULO o ESTADO: 0
🔸 Normalización Profunda Completada.
   Ejemplo: "YOGUR LIQUIDO 0% 0% CON FRESA" -> "YOGUR LIQUIDO 0% 0% CON FRESA"


## 2. Aislamiento de Conflictos de Etiquetado (Ground Truth Noise)

**Justificación (EDA 01b — Sección 9, Conflictos de Etiquetado):**
El EDA detectó descripciones idénticas catalogadas tanto como PERMITIDO como NO PERMITIDO por distintos operarios.
Estas contradicciones generan señales opuestas durante el entrenamiento del modelo, degradando su capacidad de generalización.

**Decisión de diseño:** Los conflictos se detectan **por supermercado** (`NOMBRE_EMPRESA` + `DESCRIPCION_ARTICULO`),
porque el EDA (Sección 7) confirmó que cada supermercado tiene su propia taxonomía interna.
Una misma descripción en dos supermercados distintos puede referirse a productos legítimamente diferentes.

Las filas conflictivas se **aíslan** (no se eliminan) en un fichero aparte para resolución humana.

In [3]:
# Registrar estado PRE-limpieza para validación posterior
n_total_pre = len(df)
n_np_pre = (df['ESTADO'] == 'NO PERMITIDO').sum()
n_p_pre = (df['ESTADO'] == 'PERMITIDO').sum()
ratio_pre = n_p_pre / max(n_np_pre, 1)

print(f'=== ESTADO PRE-LIMPIEZA ===')
print(f'   Total filas: {n_total_pre:,}')
print(f'   PERMITIDO: {n_p_pre:,} | NO PERMITIDO: {n_np_pre:,}')
print(f'   Ratio desbalance: {ratio_pre:.0f}:1')

# --- Detección de conflictos POR SUPERMERCADO ---
conflicto_check = df.groupby(['NOMBRE_EMPRESA', 'DESCRIPCION_ARTICULO'])['ESTADO'].nunique()
pares_conflictivos = conflicto_check[conflicto_check > 1].index

# Marcar filas conflictivas
df['_clave_conflicto'] = list(zip(df['NOMBRE_EMPRESA'], df['DESCRIPCION_ARTICULO']))
mask_conflicto = df['_clave_conflicto'].isin(pares_conflictivos)

# Extraer y exportar conflictos
df_conflictos = df[mask_conflicto][columnas_base].drop_duplicates()
file_conflictos = f'{OUTPUT_PATH}/conflictos_para_visto_bueno.csv'
df_conflictos.to_csv(file_conflictos, index=False, sep=';', encoding='utf-8')

n_desc_conflicto = len(pares_conflictivos)
n_filas_conflicto = mask_conflicto.sum()
print(f'\n\u26A0\uFE0F  {n_desc_conflicto:,} pares (Empresa+Descripción) con etiquetado contradictorio.')
print(f'   Filas afectadas: {n_filas_conflicto:,}')
print(f'   Exportado a: {file_conflictos}')

if n_desc_conflicto > 0:
    print(f'\n   Ejemplos de conflicto:')
    display(df_conflictos.head(6))

# Limpiar columna auxiliar
df = df.drop(columns=['_clave_conflicto'])

=== ESTADO PRE-LIMPIEZA ===
   Total filas: 659,272
   PERMITIDO: 656,576 | NO PERMITIDO: 2,696
   Ratio desbalance: 244:1

⚠️  0 pares (Empresa+Descripción) con etiquetado contradictorio.
   Filas afectadas: 0
   Exportado a: /content/drive/MyDrive/Colab Notebooks/TFM - Master en IA y Ciencia de Datos/data/processed/conflictos_para_visto_bueno.csv


## 3. Construcción del Diccionario Maestro (Des-duplicación)

**Justificación (EDA 01b — Sección 9, Duplicados Exactos):**
El análisis reveló que la inmensa mayoría de las filas son transacciones repetidas del mismo producto.
Pasar 50.000 veces el texto "LECHE PASCUAL ENT" por TF-IDF o un Transformer no enseña nada nuevo al modelo,
pero sí consume RAM y tiempo de cómputo (restricción crítica en Colab Free).

**Clave de des-duplicación:** `[NOMBRE_EMPRESA, DESCRIPCION_ARTICULO, ESTADO, SUBFAMILIA]`

**¿Por qué NO se incluye `COD_EAN`?**
El EDA (Sección 5) demostró que los EANs presentan irregularidades (códigos internos, nulos) y contradicciones.
Incluirlo en la clave de des-duplicación generaría duplicados falsos (mismo producto con EAN distinto según el ticket).

In [4]:
# Aislar los conflictos (NO formarán parte del corpus de entrenamiento)
df['_clave_conflicto'] = list(zip(df['NOMBRE_EMPRESA'], df['DESCRIPCION_ARTICULO']))
df_sane = df[~df['_clave_conflicto'].isin(pares_conflictivos)].drop(columns=['_clave_conflicto'])

# Des-duplicación estricta preservando la taxonomía
clave_dedup = ['NOMBRE_EMPRESA', 'DESCRIPCION_ARTICULO', 'ESTADO', 'SUBFAMILIA']
df_maestro = df_sane.drop_duplicates(subset=clave_dedup).copy()

# === MÉTRICAS DE COMPRESIÓN ===
ratio_compresion = (1 - (df_maestro.shape[0] / n_total_pre)) * 100
print(f'\n\U0001F539 Resultado del Embudamiento:')
print(f'   Transacciones brutas iniciales:  {n_total_pre:,}')
print(f'   (-) Filas conflictivas aisladas: {n_filas_conflicto:,}')
print(f'   (-) Duplicados colapsados:       {len(df_sane) - len(df_maestro):,}')
print(f'   (=) Entidades únicas resultantes: {df_maestro.shape[0]:,}')
print(f'   Reducción computacional:          {ratio_compresion:.1f}%')


🔹 Resultado del Embudamiento:
   Transacciones brutas iniciales:  659,272
   (-) Filas conflictivas aisladas: 0
   (-) Duplicados colapsados:       620,578
   (=) Entidades únicas resultantes: 38,694
   Reducción computacional:          94.1%


## 4. Validación de Integridad Post-Limpieza

**Objetivo:** Garantizar que la limpieza no destruyó información crítica de la clase minoritaria.
Si se perdieron registros NO PERMITIDO más allá de los conflictos aislados, hay un error lógico en el pipeline.

In [5]:
n_total_post = len(df_maestro)
n_np_post = (df_maestro['ESTADO'] == 'NO PERMITIDO').sum()
n_p_post = (df_maestro['ESTADO'] == 'PERMITIDO').sum()
ratio_post = n_p_post / max(n_np_post, 1)

# NP perdidos que NO son por conflictos
np_en_conflictos = (df_conflictos['ESTADO'] == 'NO PERMITIDO').sum() if len(df_conflictos) > 0 else 0
np_contabilizados = n_np_post + np_en_conflictos

print(f'=== VALIDACIÓN DE INTEGRIDAD ===')
print(f'')
print(f'                        PRE-LIMPIEZA    POST-LIMPIEZA')
print(f'   Total filas:         {n_total_pre:>12,}    {n_total_post:>12,}')
print(f'   PERMITIDO:           {n_p_pre:>12,}    {n_p_post:>12,}')
print(f'   NO PERMITIDO:        {n_np_pre:>12,}    {n_np_post:>12,}')
print(f'   Ratio desbalance:    {ratio_pre:>11.0f}:1    {ratio_post:>11.0f}:1')
print(f'')
print(f'   NP en conflictos aislados: {np_en_conflictos:,}')
print(f'   NP contabilizados (maestro + conflictos): {np_contabilizados:,}')

# Verificación de seguridad
if n_np_post == 0:
    print('\n\u274C ERROR CRÍTICO: Se han perdido TODOS los registros NO PERMITIDO.')
elif n_np_post < 100:
    print(f'\n\u26A0\uFE0F  ADVERTENCIA: Solo quedan {n_np_post} registros NO PERMITIDO únicos. Verificar manualmente.')
else:
    print(f'\n\u2705 Integridad verificada. La clase minoritaria retiene {n_np_post:,} entidades únicas.')

=== VALIDACIÓN DE INTEGRIDAD ===

                        PRE-LIMPIEZA    POST-LIMPIEZA
   Total filas:              659,272          38,694
   PERMITIDO:                656,576          37,863
   NO PERMITIDO:               2,696             831
   Ratio desbalance:            244:1             46:1

   NP en conflictos aislados: 0
   NP contabilizados (maestro + conflictos): 831

✅ Integridad verificada. La clase minoritaria retiene 831 entidades únicas.


## 5. Exportación del Diccionario Maestro

El fichero resultante es la **entrada oficial** para los notebooks de modelado.
Se exporta con separador `;` para mantener consistencia con el formato origen del dataset.
Se persiste en **Google Drive** para evitar pérdida por reinicio de runtime.

In [6]:
output_maestro = f'{OUTPUT_PATH}/diccionario_maestro.csv'
df_maestro.to_csv(output_maestro, index=False, sep=';', encoding='utf-8')

print(f'\u2705 Exportado: {output_maestro}')
print(f'   Filas: {len(df_maestro):,} | Columnas: {df_maestro.shape[1]}')
print(f'   Separador: ";" (consistente con Dataset.csv)')

✅ Exportado: /content/drive/MyDrive/Colab Notebooks/TFM - Master en IA y Ciencia de Datos/data/processed/diccionario_maestro.csv
   Filas: 38,694 | Columnas: 4
   Separador: ";" (consistente con Dataset.csv)


## 6. Extracción de Reglas Deterministas (Bloqueo Léxico — Fase 1)

**Justificación (EDA 01b — Sección 6, Taxonomía):**
El análisis taxonómico reveló subfamilias donde el **100% de los artículos son NO PERMITIDO**.
Estos casos no requieren Machine Learning: son reglas de negocio puras que se pueden aplicar determinísticamente.

Adicionalmente, el experto de dominio declaró las siguientes **categorías prohibidas por ley**:
agua, bebidas alcohólicas, suavizantes, quitamanchas y ropa.

Se generan dos tipos de reglas:
1. **Reglas taxonómicas:** Subfamilias con tasa NP = 100%, segregadas por supermercado.
2. **Reglas léxicas:** Palabras clave prohibidas que activan bloqueo directo sobre `DESCRIPCION_ARTICULO`.

In [7]:
# === REGLAS TAXONÓMICAS: Subfamilias 100% NO PERMITIDO por supermercado ===
reglas_taxonomicas = {}

for empresa in df_sane['NOMBRE_EMPRESA'].unique():
    df_emp = df_sane[df_sane['NOMBRE_EMPRESA'] == empresa]
    # Tasa NP por subfamilia dentro de este supermercado
    tasa = df_emp.groupby('SUBFAMILIA')['ESTADO'].apply(
        lambda x: (x == 'NO PERMITIDO').mean()
    )
    # Subfamilias con 100% NP (regla dura irrefutable)
    subfam_bloqueadas = tasa[tasa == 1.0].index.tolist()
    # Subfamilias con tasa NP > 50% pero < 100% (candidatas a revisión)
    subfam_sospechosas = tasa[(tasa > 0.5) & (tasa < 1.0)].index.tolist()

    reglas_taxonomicas[empresa] = {
        'lista_negra_subfamilias': subfam_bloqueadas,
        'subfamilias_sospechosas': subfam_sospechosas
    }
    print(f'\n{empresa}:')
    print(f'   Lista Negra (100% NP): {len(subfam_bloqueadas)} subfamilias -> {subfam_bloqueadas[:5]}')
    print(f'   Sospechosas (>50% NP): {len(subfam_sospechosas)} subfamilias -> {subfam_sospechosas[:5]}')

# === REGLAS LÉXICAS: Palabras clave prohibidas (declaradas por el experto de dominio) ===
palabras_prohibidas = [
    'ALCOHOL', 'ALCOHOLIC', 'CERVEZA', 'VINO', 'VODKA', 'RON', 'WHISKY', 'GINEBRA', 'LICOR',
    'SUAVIZANTE', 'QUITAMANCHAS', 'LEJIA',
    'ROPA', 'CAMISETA', 'PANTALON', 'CALCETINES',
]

reglas_completas = {
    'reglas_taxonomicas_por_supermercado': reglas_taxonomicas,
    'palabras_prohibidas_lexicas': palabras_prohibidas
}

# Exportar a JSON
output_reglas = f'{OUTPUT_PATH}/reglas_deterministas.json'
with open(output_reglas, 'w', encoding='utf-8') as f:
    json.dump(reglas_completas, f, ensure_ascii=False, indent=2)

print(f'\n\u2705 Reglas deterministas exportadas a: {output_reglas}')
print(f'   Palabras prohibidas léxicas: {len(palabras_prohibidas)}')


SUPERMERCADO1:
   Lista Negra (100% NP): 48 subfamilias -> ['Accesorios y útiles perfumería', 'Aditivos para lavado', 'Aerosol', 'Alimentación deportiva', 'Aluminio']
   Sospechosas (>50% NP): 1 subfamilias -> ['Sin gas']

SUPERMERCADO3:
   Lista Negra (100% NP): 2 subfamilias -> ['COLONIAS Y PERFUMES', 'VINOS']
   Sospechosas (>50% NP): 0 subfamilias -> []

SUPERMERCADO2:
   Lista Negra (100% NP): 3 subfamilias -> ['APERITIVOS', 'CREMAS LIQUIDAS', 'PAPEL HIGIENICO HUMEDO']
   Sospechosas (>50% NP): 0 subfamilias -> []

SUPERMERCADO4:
   Lista Negra (100% NP): 46 subfamilias -> ['ADORNOS', 'ARTESANAS /PAJA', 'BARQUILLOS', 'BASICAS', 'BERLINAS']
   Sospechosas (>50% NP): 2 subfamilias -> ['CARNES PREPAS', 'ZUMOS']

SUPERMERCADO5:
   Lista Negra (100% NP): 24 subfamilias -> ['APERITIVOS  EMBUTIDOS', 'BOLSAS CAMISETAS', 'CARNES EMPANADAS CONGELADAS', 'CERDO IBERICO', 'CHAMPU SECO']
   Sospechosas (>50% NP): 13 subfamilias -> ['AYUDAS CULINARIAS', 'BEBIDA FRESCA CAFE OTROS', 'COMPOTAS-SMO

## 7. Conclusión y Resumen de la Fase de Preparación

### Logros Cuantificables
La depuración inteligente ha conseguido tres hitos operativos críticos:

1. **Reducción masiva del volumen computacional (~94%):** De ~659.000 transacciones brutas a ~38.000 entidades únicas. Esto hace viable el entrenamiento completo en Google Colab Free sin necesidad de under-sampling artificial destructivo.

2. **Mejora drástica del ratio de desbalance:** De **251:1** a **~47:1**. Este nuevo ratio, combinado con `class_weight='balanced'`, sitúa el problema en un rango gestionable por clasificadores lineales estándar.

3. **Extracción de reglas deterministas:** Se han generado diccionarios JSON de bloqueo léxico por supermercado, completando la Fase 1 del pipeline híbrido.

### Artefactos Generados

| Fichero | Descripción | Ubicación |
|---|---|---|
| `diccionario_maestro.csv` | Corpus limpio para entrenamiento NLP | `data/processed/` |
| `conflictos_para_visto_bueno.csv` | Descripciones con etiquetado contradictorio | `data/processed/` |
| `reglas_deterministas.json` | Listas Negras taxonómicas + palabras prohibidas | `data/processed/` |

---
**Siguiente paso:** `02b_data_understanding_post_prep.ipynb` (EDA del Diccionario Maestro para validación visual ante el tribunal).